### RAG Pipeline - Data Ingestion to Vector DB Pipeline


In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/tmp/ipykernel_106611/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all pdf files from a directory


def load_single_pdf(pdf_file: Path):
    """Loads one PDF, falling back to PyMuPDFLoader if PyPDFLoader fails."""
    try:
        documents = PyPDFLoader(str(pdf_file)).load()
        return documents, "PyPDFLoader"
    except Exception as e:
        print(f"  PyPDFLoader failed on {pdf_file.name}: {e}. Falling back to PyMuPDFLoader.")
        documents = PyMuPDFLoader(str(pdf_file)).load()
        return documents, "PyMuPDFLoader"


def process_all_pdfs_in_directory(directory_path):
    all_documents = []
    pdf_directory = Path(directory_path)

    # rglob on a missing directory returns nothing instead of failing, which looks
    # exactly like an empty corpus - check explicitly so a bad path is obvious.
    if not pdf_directory.is_dir():
        raise NotADirectoryError(
            f"PDF directory not found: '{pdf_directory.resolve()}' "
            f"(paths here are relative to the notebook's working directory)."
        )

    # rglob + lowercased suffix so nested folders and .PDF files are not skipped
    pdf_files = sorted(
        f for f in pdf_directory.rglob("*") if f.is_file() and f.suffix.lower() == ".pdf"
    )

    print(f"Found {len(pdf_files)} PDF files in the directory '{directory_path}'.")

    failed_files = []
    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file.name}")
        try:
            documents, loader_used = load_single_pdf(pdf_file)

            if not documents:
                print(f"  Warning: {pdf_file.name} produced 0 pages (scanned PDF with no text layer?).")

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = pdf_file.suffix.lower()
            all_documents.extend(documents)
            print(
                f"Successfully loaded {len(documents)} pages from {pdf_file.name} using {loader_used}."
            )

        except Exception as e:
            failed_files.append(pdf_file.name)
            print(f"  Skipping {pdf_file.name}: both loaders failed ({e}).")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    if failed_files:
        print(f"Failed files ({len(failed_files)}): {failed_files}")
    return all_documents


all_documents = process_all_pdfs_in_directory("../data/pdf_files")

Found 1 PDF files in the directory '../data/pdf_files'.
Processing file: Daniel-Shiffman_Nature-of-code.pdf
Successfully loaded 521 pages from Daniel-Shiffman_Nature-of-code.pdf using PyPDFLoader.

Total documents loaded: 521


In [3]:
all_documents

[Document(metadata={'producer': 'Prince 8.1 rev 3 (www.princexml.com)', 'creator': 'PyPDF', 'creationdate': '2017-02-13T11:22:15-02:00', 'moddate': '2017-02-13T11:22:15-02:00', 'title': 'The Nature of Code', 'source': '../data/pdf_files/Daniel-Shiffman_Nature-of-code.pdf', 'total_pages': 521, 'page': 0, 'page_label': '1', 'source_file': 'Daniel-Shiffman_Nature-of-code.pdf', 'file_type': '.pdf'}, page_content='T HE NATURE OF CODE\nT HE NATURE OF CODE\nSIMULATING NATURAL SYSTEMS WITH PROCESSING\nD ANIEL SHIFFMAN\nTHE NATURE OF CODE\nDANIEL SHIFFMAN\nHow can we capture the unpredictable \nevolutionary and emergent properties of \nnature in software?\nHow can understanding the mathematical \nprinciples behind our physical world help us \nto create digital worlds?\nThis book focuses on a range of \nprogramming strategies and techniques \nbehind computer simulations of natural \nsystems, from elementary concepts in \nmathematics and physics to more advanced \nalgorithms that enable sophistic

In [4]:
### Text Splitter - Turning documents into chunks for embedding and storage in vector database

def text_splitter(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len, separators=["\n\n", "\n", " ", ""]
    )
    split_documents = text_splitter.split_documents(documents)
    print(f"Total chunks created: {len(split_documents)}")

    # Example chunk
    if split_documents:
        print(f"Example chunk:\n{split_documents[0].page_content[:500]}...")
        print(f"Metadata of the example chunk:\n{split_documents[0].metadata}")

    return split_documents

In [5]:
chunks = text_splitter(all_documents)
chunks

Total chunks created: 1089
Example chunk:
T HE NATURE OF CODE
T HE NATURE OF CODE
SIMULATING NATURAL SYSTEMS WITH PROCESSING
D ANIEL SHIFFMAN
THE NATURE OF CODE
DANIEL SHIFFMAN
How can we capture the unpredictable 
evolutionary and emergent properties of 
nature in software?
How can understanding the mathematical 
principles behind our physical world help us 
to create digital worlds?
This book focuses on a range of 
programming strategies and techniques 
behind computer simulations of natural 
systems, from elementary concepts in 
math...
Metadata of the example chunk:
{'producer': 'Prince 8.1 rev 3 (www.princexml.com)', 'creator': 'PyPDF', 'creationdate': '2017-02-13T11:22:15-02:00', 'moddate': '2017-02-13T11:22:15-02:00', 'title': 'The Nature of Code', 'source': '../data/pdf_files/Daniel-Shiffman_Nature-of-code.pdf', 'total_pages': 521, 'page': 0, 'page_label': '1', 'source_file': 'Daniel-Shiffman_Nature-of-code.pdf', 'file_type': '.pdf'}


[Document(metadata={'producer': 'Prince 8.1 rev 3 (www.princexml.com)', 'creator': 'PyPDF', 'creationdate': '2017-02-13T11:22:15-02:00', 'moddate': '2017-02-13T11:22:15-02:00', 'title': 'The Nature of Code', 'source': '../data/pdf_files/Daniel-Shiffman_Nature-of-code.pdf', 'total_pages': 521, 'page': 0, 'page_label': '1', 'source_file': 'Daniel-Shiffman_Nature-of-code.pdf', 'file_type': '.pdf'}, page_content='T HE NATURE OF CODE\nT HE NATURE OF CODE\nSIMULATING NATURAL SYSTEMS WITH PROCESSING\nD ANIEL SHIFFMAN\nTHE NATURE OF CODE\nDANIEL SHIFFMAN\nHow can we capture the unpredictable \nevolutionary and emergent properties of \nnature in software?\nHow can understanding the mathematical \nprinciples behind our physical world help us \nto create digital worlds?\nThis book focuses on a range of \nprogramming strategies and techniques \nbehind computer simulations of natural \nsystems, from elementary concepts in \nmathematics and physics to more advanced \nalgorithms that enable sophistic

### Embeddings & Vector DB

In [6]:
import hashlib
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from typing import List, Dict, Any
from langchain_core.documents import Document

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initializes the EmbeddingManager with a specified model."""
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads the Sentence Transformer model."""
        print(f"Loading embedding model: {self.model_name}")
        # Let load failures raise: a half-built manager only fails later, more confusingly.
        self.model = SentenceTransformer(self.model_name)
        self.embedding_dim = self.model.get_embedding_dimension()
        print(
            f"Model {self.model_name} loaded successfully. Embedding dimension: {self.embedding_dim}"
        )

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generates embeddings for a list of texts.
        args:
          texts: List of strings to be embedded.
        output:
          A numpy array of embeddings corresponding to the input texts.
        """
        if not texts:
            return np.empty((0, self.embedding_dim), dtype=np.float32)
        print(f"Generating embeddings for {len(texts)} texts.")
        embeddings = self.model.encode(
            texts,
            convert_to_numpy=True,
            show_progress_bar=True,
            normalize_embeddings=True,  # unit vectors -> cosine distance is well behaved
        )
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model all-MiniLM-L6-v2 loaded successfully. Embedding dimension: 384


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings and provides methods for similarity search using ChromaDB."""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store",
    ):
        """Initializes the VectorStore with a specified collection name and persistence directory."""
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        """Initializes the ChromaDB client and collection."""
        print(
            f"Initializing ChromaDB client with persistence directory: {self.persist_directory}"
        )
        os.makedirs(self.persist_directory, exist_ok=True)
        # Let failures raise: swallowing them leaves self.collection = None and the real
        # error only surfaces much later, far from its cause.
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            # Default space is L2. Embeddings here are normalized, so cosine is the
            # right metric and makes `1 - distance` a real similarity in [0, 1].
            configuration={"hnsw": {"space": "cosine"}},
            metadata={"description": "Collection of PDF document embeddings"},
        )
        print(f"ChromaDB collection '{self.collection_name}' initialized successfully.")
        print(f"Existing documents in the collection: {self.collection.count()} documents.")

    @staticmethod
    def _make_id(doc: Document, seen: Dict[str, int]) -> str:
        """Builds a stable ID from the chunk's origin and content.

        Deterministic IDs make ingestion idempotent: re-running upserts over the same
        rows instead of inserting near-identical copies under fresh random IDs.
        """
        source = doc.metadata.get("source_file") or doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "na")
        key = f"{source}|{page}|{doc.page_content}"
        digest = hashlib.sha256(key.encode("utf-8")).hexdigest()[:16]
        # Guard the rare case of byte-identical chunks on the same page.
        occurrence = seen.get(digest, 0)
        seen[digest] = occurrence + 1
        return f"{digest}_{occurrence}"

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        """Adds documents and their embeddings to the ChromaDB collection.
        args:
          documents: List of LangChain Document chunks.
          embeddings: Numpy array of embeddings corresponding to the documents.
        """
        if self.collection is None:
            raise ValueError("ChromaDB collection not initialized.")
        if len(documents) != embeddings.shape[0]:
            raise ValueError(
                f"Number of documents ({len(documents)}) and embeddings "
                f"({embeddings.shape[0]}) must match."
            )
        if not documents:
            print("No documents to add.")
            return

        # Preparing data for ChromaDB

        ids = []
        metadatas = []
        documents_content = []
        embeddings_list = []
        seen: Dict[str, int] = {}

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            ids.append(self._make_id(doc, seen))

            metadata = dict(doc.metadata)  # copy, don't mutate the chunk
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        # Chroma caps how many rows one call may carry, so write in batches.
        batch_size = min(getattr(self.client, "get_max_batch_size", lambda: 5000)(), 5000)
        for start in range(0, len(ids), batch_size):
            end = start + batch_size
            self.collection.upsert(
                ids=ids[start:end],
                metadatas=metadatas[start:end],
                embeddings=embeddings_list[start:end],
                documents=documents_content[start:end],
            )

        print(
            f"Upserted {len(documents)} documents into '{self.collection_name}'. "
            f"Collection now holds {self.collection.count()} documents."
        )

    def reset(self):
        """Drops and recreates the collection - useful after a bad ingestion run."""
        self.client.delete_collection(self.collection_name)
        self.collection = None
        self._initialize_chromadb()

vector_store = VectorStore()
vector_store

Initializing ChromaDB client with persistence directory: ../data/vector_store
ChromaDB collection 'pdf_documents' initialized successfully.
Existing documents in the collection: 0 documents.


In [9]:
### Convert the texts into embeddings and store them in the vector database
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 1089 texts.


Batches:   0%|          | 0/35 [00:00<?, ?it/s]

Generated embeddings with shape: (1089, 384)
Upserted 1089 documents into 'pdf_documents'. Collection now holds 1089 documents.



### Retriever Pipeline from VectorStore

In [10]:
class RAGRetriever:
    """Retriever for fetching relevant documents from the vector store based on a query."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initializes the RAGRetriever with a specified VectorStore."""
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """Retrieves the top_k most relevant documents for a given query.
        args:
          query: The input query string.
          top_k: The number of top documents to retrieve.
          score_threshold: Minimum cosine similarity (0-1) to consider a document relevant.
        output:
          A list of dictionaries containing the retrieved documents and their metadata.
        """
        if not query or not query.strip():
            raise ValueError("Query must be a non-empty string.")
        if self.vector_store.collection is None:
            raise ValueError("ChromaDB collection not initialized.")

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        # Same embedding model as ingestion - vectors from different models aren't comparable.
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Errors propagate: an empty list here is indistinguishable from "no matches",
        # which is how a dead collection previously looked like a valid empty result.
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
            include=["documents", "metadatas", "distances"],
        )

        retrieved_docs = []

        if results["documents"] and results["documents"][0]:
            documents = results["documents"][0]
            metadatas = results["metadatas"][0]
            distances = results["distances"][0]
            ids = results["ids"][0]

            for i, (doc, metadata, distance, doc_id) in enumerate(
                zip(documents, metadatas, distances, ids)
            ):
                # Collection uses cosine space, so distance is in [0, 2]
                # and 1 - distance is a genuine cosine similarity.
                similarity_score = 1 - distance
                if similarity_score >= score_threshold:
                    retrieved_docs.append(
                        {
                            "id": doc_id,
                            "content": doc,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1,
                        }
                    )
            print(
                f"Retrieved {len(retrieved_docs)} documents above the score threshold of {score_threshold}."
            )
        else:
            print("No documents retrieved from the vector store.")

        return retrieved_docs


rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever

In [11]:
rag_retriever.retrieve("What is an LLM?")

Retrieving documents for query: 'What is an LLM?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents above the score threshold of 0.0.


[{'id': '6ea67910a836da41_0',
  'content': '• VerletConstrainedSpring : A VerletConstrainedSpring object is a spring whose\nmaximum distance can be limited. This can help the whole spring system achieve\nbetter stability.\n• VerletMinDistanceSpring : A VerletMinDistanceSpring object is a spring that\nonly enforces its rest length if the current distance is less than its rest length. This is\nhandy if you want to ensure objects are at least a certain distance from each other,\nbut don’t care if the distance is bigger than the enforced minimum.\nThe inheritance and polymorphism technique we employed in the previous section also\nproves to be useful when creating springs. A spring expects two particles when it is created.\nAnd again, because our Particle class extends VerletParticle , a VerletSpring object\nwill accept our Particle objects passed into the constructor. Let’s take a look at some\nexample code that assumes the existence of our two previous particles p1 and p2 and creates',
 

In [12]:
rag_retriever.retrieve("What is GELU?")

Retrieving documents for query: 'What is GELU?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents above the score threshold of 0.0.


[{'id': '3634a375d2b5a91e_0',
  'content': '6.2 Vehicles and Steering6.2 Vehicles and Steering\nNow that we understand the core concepts behind autonomous agents, we can begin\nwriting the code. There are many places where we could start. Artificial simulations of ant\nand termite colonies are fantastic demonstrations of systems of autonomous agents. (For\nmore on this topic, I encourage you to read Turtles, Termites, and Traffic Jams by Mitchel\nResnick.) However, we want to begin by examining agent behaviors that build on the work\nwe’ve done in the first five chapters of this book: modeling motion with vectors and driving\nmotion with forces. And so it’s time to rename our Mover class that became our Particle\nclass once again. This time we are going to call it Vehicle.\nIn his 1999 paper “Steering Behaviors for Autonomous Characters,” Reynolds uses the word\n“vehicle” to describe his autonomous agents, so we will follow suit.\nWhy Vehicle?Why Vehicle?\nIn 1986, Italian neuroscienti

### Integration VectorDB Context Pipeline with LLMs Output

In [13]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm_client = ChatGroq(
    api_key=groq_api_key, model="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024
)

#### Simple RAG

In [ ]:
def rag_simple(query, retreiver, llm_client, top_k=3):
    results = retreiver.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results] if results else "")
    if not context:
        return "No relevant documents found for the query."

    prompt = f"Answer the following question based on the context provided:\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    response = llm_client.invoke([prompt.format(context=context, query=query)])
    return response.content

In [16]:
answer = rag_simple("What are the Core elements of a Box2D world?", rag_retriever, llm_client, top_k=3)
print(f"Answer:\n{answer}")

Retrieving documents for query: 'What are the Core elements of a Box2D world?'
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the score threshold of 0.0.
Answer:
The Core elements of a Box2D world are:

1. World (World): Manages the physics simulation.
2. Body (Body): Serves as the primary element in the Box2D world, having a location and a velocity.
3. Shape (Shape): Keeps track of all the necessary collision geometry attached to a body.
4. Fixture (Fixture): Attaches a shape to a body and sets properties such as density, friction, and restitution.


#### Advanced RAG

In [18]:
def rag_advanced(
    query, retriever, llm_client, top_k=3, min_score=0.2, return_context=False
):
    """RAG Pipeline with advanced options for top_k, minimum score threshold, and optional context return."""
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return "No relevant documents found for the query."

    context = "\n\n".join([doc["content"] for doc in results])

    sources = [
        {
            "source": doc["metadata"].get(
                "source_file", doc["metadata"].get("source", "unknown")
            ),
            "page": doc["metadata"].get("page", "unknown"),
            "score": doc["similarity_score"],
            "preview": (
                doc["content"][:300] + "..."
                if len(doc["content"]) > 300
                else doc["content"]
            ),
        }
        for doc in results
    ]
    confidence = max([doc["similarity_score"] for doc in results])

    prompt = f"""
    You are an AI assistant. Use the following context to answer the question below.

    Context:
    {context}

    Question: {query}

    Answer in a concise and informative manner.
    """

    response = llm_client.invoke([prompt.format(context=context, query=query)])
    output = {"answer": response.content, "sources": sources, "confidence": confidence}

    if return_context:
        output["context"] = context
    return output


result = rag_advanced(
    "What are the Core elements of a Box2D world?",
    rag_retriever,
    llm_client,
    top_k=3,
    min_score=0.2,
    return_context=True,
)

print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")
print(f"Confidence: {result['confidence']}")
print(f"Context: {result['context'][:300]}")

Retrieving documents for query: 'What are the Core elements of a Box2D world?'
Top K: 3, Score Threshold: 0.2
Generating embeddings for 1 texts.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the score threshold of 0.2.
Answer: The core elements of a Box2D world are:

1. **World**: Manages the physics simulation and stores lists of every element in the world.
2. **Body**: The primary element in the Box2D world, representing an object with a location and velocity.
3. **Shape**: Keeps track of the collision geometry attached to a body.
4. **Fixture**: Attaches a shape to a body and sets properties such as density, friction, and restitution.
Sources: [{'source': 'Daniel-Shiffman_Nature-of-code.pdf', 'page': 215, 'score': 0.6056698560714722, 'preview': '2. Translate the pixel world into the Box2D world.\nDRAWDRAW\n1. Ask Box2D where everything is.\n2. Translate Box2D’s answer into the pixel world.\n3. Draw everything.\nNow that we understand that anything we create in our Processing sketch has to be placed\ninto the Box2D world, let’s look at an overview...'}, {'source': 'Daniel-Shiffman_Nature-of-code.pdf', 

In [24]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm_client)
result = adv_rag.query("What are the Core elements of a Box2D world?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What are the Core elements of a Box2D world?'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the score threshold of 0.1.
Streaming answer:
Use the following context to answer the question concisely.
Context:
2. Translate the pixel world into the Box2D world.
DRAWDRAW
1. Ask Box2D where everything is.
2. Translate Box2D’s answer into the pixel world.
3. Draw everything.
Now that we understand that anything we create in our Processing sketch has to be placed
into the Box2D world, let’s look at an overview of the elements that make up that world.
Core elements of a Box2D world:Core elements of a Box2D world:
1. WorldWorld : Manages the physics simulation. It knows everything about the overall
coordinate space and also stores lists of every element in the world (see 2-4
below).
2. BodyBody: Serves as the primary element in the Box2D world. It has a location. It has a
velocity. Sound familiar? The Body is essentially the class we’ve been writing on
our own in our vectors and forces examples.
3. ShapeShape : Keeps